# DS102 v5 — không dùng pretrained

Chạy trên **hosted Google Colab**, CPU đủ dùng. Tất cả mô hình học từ train;
không dùng pretrained embedding, pretrained model hoặc dữ liệu ngoài cuộc thi.
Thư viện: scikit-learn (BSD-3-Clause), CatBoost (Apache-2.0), và fastText (MIT)
nếu cấu hình được chọn sử dụng fastText. fastText được biên dịch từ mã nguồn
chính thức v0.9.2; không tải pretrained model hoặc pretrained vector.

Upload `colab_data.zip` hoặc bốn CSV. Notebook chứa toàn bộ mã pipeline và
huấn luyện lại từ đầu, không phụ thuộc cache dự đoán của lần thử trước.
Cấu hình đã chọn bằng OOF trên train; notebook này không chạy lại tìm kiếm CV.
Điểm Kaggle chỉ biết sau khi nộp CSV, không thể bảo đảm tăng điểm.


In [ ]:
import os,platform,sys,subprocess
assert platform.system()=='Linux' and os.environ.get('COLAB_RELEASE_TAG'), 'Use hosted Colab, not local runtime'
os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='2'
subprocess.check_call([sys.executable,'-m','pip','install','-q','scikit-learn==1.7.2','catboost==1.2.10'])
from threadpoolctl import threadpool_limits
threadpool_limits(2)


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded=files.upload()
data_dir=Path('/content/ds102_data');data_dir.mkdir(exist_ok=True)
required={'ds102_train.csv','ds102_dev.csv','ds102_test.csv','ds102_samplesubmission.csv'}
for name,content in uploaded.items():
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            for member in z.namelist():
                if Path(member).name in required:
                    (data_dir/Path(member).name).write_bytes(z.read(member))
    elif Path(name).name in required:
        (data_dir/Path(name).name).write_bytes(content)
assert all((data_dir/n).exists() for n in required),'Missing one of four CSVs'


In [ ]:
import json,sys
from pathlib import Path
source=Path("/content/ds102_v5_source");source.mkdir(exist_ok=True)
modules=json.loads('{"train_submit.py": "\\"\\"\\"CPU baseline cho DS102: TF-IDF + Linear SVM, chọn theo F1-macro.\\n\\nChỉ dùng comment và nhãn train. Dev được đánh giá sau khi chọn tham số\\nbằng cross-validation trên train; không gộp dev vào train.\\n\\"\\"\\"\\n\\nimport argparse\\nimport html\\nimport json\\nimport platform\\nimport re\\nimport unicodedata\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport sklearn\\nfrom scipy import sparse\\nfrom sklearn.base import BaseEstimator, TransformerMixin\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import LinearSVC\\n\\n\\nSEED = 102\\nLABELS = [0, 1, 2, 3]\\nFILES = {name: f\\"ds102_{name}.csv\\" for name in [\\"train\\", \\"dev\\", \\"test\\", \\"samplesubmission\\"]}\\n\\n\\ndef normalize_text(value):\\n    \\"\\"\\"Giữ dấu, emoji, dấu câu và ký tự lặp vì có thể giúp nhận diện spam.\\"\\"\\"\\n    text = html.unescape(unicodedata.normalize(\\"NFC\\", str(value)))\\n    text = text.replace(\\"\\\\\\\\r\\\\\\\\n\\", \\"\\\\n\\").replace(\\"\\\\\\\\n\\", \\"\\\\n\\").replace(\\"\\\\\\\\r\\", \\"\\\\n\\")\\n    return re.sub(r\\"\\\\s+\\", \\" \\", text).strip().lower()\\n\\n\\nclass ReviewStats(BaseEstimator, TransformerMixin):\\n    \\"\\"\\"Đặc trưng độ dài, lặp ký tự và tỷ lệ ký tự; không có từ khóa nhãn.\\"\\"\\"\\n\\n    def fit(self, X, y=None):\\n        return self\\n\\n    def transform(self, X):\\n        rows = []\\n        for text in X:\\n            chars = len(text)\\n            words = text.split()\\n            nwords = len(words)\\n            denom = max(chars, 1)\\n            word_denom = max(nwords, 1)\\n            letters = [c for c in text if c.isalpha()]\\n            longest_run = max((len(m.group()) for m in re.finditer(r\\"(.)\\\\1*\\", text)), default=0)\\n            rows.append([\\n                np.log1p(chars), np.log1p(nwords), chars / word_denom,\\n                len(set(words)) / word_denom, len(set(text)) / denom,\\n                longest_run / denom, np.log1p(longest_run),\\n                sum(c.isdigit() for c in text) / denom,\\n                len(letters) / denom,\\n                sum(not c.isalnum() and not c.isspace() for c in text) / denom,\\n                sum(ord(c) > 0xFFFF for c in text) / denom,\\n                text.count(\\"!\\") / denom, text.count(\\"?\\") / denom,\\n                float(bool(re.search(r\\"https?://|www\\\\.\\", text))),\\n            ])\\n        return sparse.csr_matrix(np.asarray(rows, dtype=np.float32))\\n\\n\\ndef make_features():\\n    return FeatureUnion([\\n        (\\"word\\", TfidfVectorizer(\\n            ngram_range=(1, 2), min_df=2, max_df=1.0, max_features=90000,\\n            sublinear_tf=True, lowercase=False, dtype=np.float32,\\n            token_pattern=r\\"(?u)\\\\b\\\\w+\\\\b\\",\\n        )),\\n        (\\"char\\", TfidfVectorizer(\\n            analyzer=\\"char\\", ngram_range=(2, 5), min_df=3, max_features=160000,\\n            sublinear_tf=True, lowercase=False, dtype=np.float32,\\n        )),\\n        (\\"stats\\", Pipeline([\\n            (\\"extract\\", ReviewStats()), (\\"scale\\", StandardScaler(with_mean=False)),\\n        ])),\\n    ], transformer_weights={\\"word\\": 1.0, \\"char\\": 1.0, \\"stats\\": 0.1})\\n\\n\\ndef make_model(config, y):\\n    weight = config[\\"weight\\"]\\n    if weight == \\"sqrt_balanced\\":\\n        counts = pd.Series(y).value_counts()\\n        weight = {int(k): float(np.sqrt(len(y) / (len(counts) * v))) for k, v in counts.items()}\\n    elif weight == \\"none\\":\\n        weight = None\\n    return LinearSVC(C=config[\\"C\\"], class_weight=weight, dual=True,\\n                     max_iter=10000, tol=1e-4, random_state=SEED)\\n\\n\\ndef find_data_dir(supplied=None):\\n    if supplied:\\n        path = Path(supplied).expanduser().resolve()\\n        if all((path / f).is_file() for f in FILES.values()):\\n            return path\\n        raise FileNotFoundError(f\\"Thiếu một trong 4 file CSV tại {path}\\")\\n    candidates = [Path.cwd(), Path.cwd() / \\"data\\", Path.cwd() / \\"dataset\\"]\\n    kaggle = Path(\\"/kaggle/input\\")\\n    if kaggle.exists():\\n        candidates += sorted({p.parent for p in kaggle.rglob(FILES[\\"train\\"])})\\n    matches = [p for p in candidates if all((p / f).is_file() for f in FILES.values())]\\n    if len(matches) == 1:\\n        return matches[0].resolve()\\n    raise FileNotFoundError(\\"Hãy truyền --data-dir đến thư mục chứa đúng 4 CSV; trên Kaggle dùng Add Input trước.\\")\\n\\n\\ndef load_data(data_dir):\\n    data = {name: pd.read_csv(data_dir / file, keep_default_na=False) for name, file in FILES.items()}\\n    for name in [\\"train\\", \\"dev\\", \\"test\\"]:\\n        required = {\\"comment\\", \\"spam_label\\"} if name != \\"test\\" else {\\"comment\\", \\"id\\"}\\n        if not required.issubset(data[name].columns):\\n            raise ValueError(f\\"{name}: thiếu cột {required - set(data[name].columns)}\\")\\n        if data[name].empty:\\n            raise ValueError(f\\"{name} rỗng\\")\\n    for name in [\\"train\\", \\"dev\\"]:\\n        values = pd.to_numeric(data[name][\\"spam_label\\"], errors=\\"raise\\")\\n        if not values.isin(LABELS).all():\\n            raise ValueError(f\\"{name}: nhãn phải thuộc {LABELS}\\")\\n        data[name][\\"spam_label\\"] = values.astype(int)\\n    if set(data[\\"train\\"][\\"spam_label\\"]) != set(LABELS):\\n        raise ValueError(\\"Train phải có đủ 4 nhãn\\")\\n    sample = data[\\"samplesubmission\\"]\\n    if list(sample.columns) != [\\"ID\\", \\"spam_label\\"]:\\n        raise ValueError(\\"Sample phải có hai cột ID,spam_label\\")\\n    test_ids, sample_ids = data[\\"test\\"][\\"id\\"].astype(str), sample[\\"ID\\"].astype(str)\\n    if not test_ids.is_unique or not sample_ids.is_unique or set(test_ids) != set(sample_ids):\\n        raise ValueError(\\"ID trong test và sample phải duy nhất và trùng tập hợp\\")\\n    return data\\n\\n\\ndef metrics(y, pred):\\n    return {\\n        \\"macro_f1\\": float(f1_score(y, pred, labels=LABELS, average=\\"macro\\", zero_division=0)),\\n        \\"accuracy\\": float(accuracy_score(y, pred)),\\n        \\"classification_report\\": classification_report(y, pred, labels=LABELS, output_dict=True, zero_division=0),\\n        \\"confusion_matrix\\": confusion_matrix(y, pred, labels=LABELS).tolist(),\\n    }\\n\\n\\ndef main(argv=None):\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--data-dir\\", default=None)\\n    parser.add_argument(\\"--output-dir\\", default=\\"output\\")\\n    parser.add_argument(\\"--folds\\", type=int, default=3)\\n    args = parser.parse_args(argv)\\n    if args.folds < 2:\\n        parser.error(\\"--folds phải >= 2\\")\\n    data_dir = find_data_dir(args.data_dir)\\n    output = Path(args.output_dir).expanduser().resolve()\\n    output.mkdir(parents=True, exist_ok=True)\\n    data = load_data(data_dir)\\n    train, dev, test = [data[n] for n in [\\"train\\", \\"dev\\", \\"test\\"]]\\n    texts = train[\\"comment\\"].map(normalize_text).to_numpy()\\n    dev_texts = dev[\\"comment\\"].map(normalize_text).to_numpy()\\n    test_texts = test[\\"comment\\"].map(normalize_text).to_numpy()\\n    y = train[\\"spam_label\\"].to_numpy()\\n    if train.groupby(train[\\"comment\\"].map(normalize_text))[\\"spam_label\\"].first().value_counts().min() < args.folds:\\n        raise ValueError(\\"Không đủ nhóm cho số folds đã chọn\\")\\n    print(f\\"Train={len(train)}, dev={len(dev)}, test={len(test)}\\", flush=True)\\n    print(f\\"Nhãn train: {train.spam_label.value_counts().sort_index().to_dict()}\\", flush=True)\\n    configs = [{\\"C\\": c, \\"weight\\": weight} for weight in [\\"none\\", \\"sqrt_balanced\\", \\"balanced\\"] for c in [1.0, 4.0]]\\n    oof = np.full((len(configs), len(y)), -1, dtype=np.int64)\\n    fold_scores = [[] for _ in configs]\\n    splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=SEED)\\n    for fold, (fit_idx, val_idx) in enumerate(splitter.split(texts, y, groups=texts), 1):\\n        print(f\\"Fold {fold}/{args.folds}: fit TF-IDF chỉ trên train của fold...\\", flush=True)\\n        features = make_features()\\n        X_fit = features.fit_transform(texts[fit_idx])\\n        X_val = features.transform(texts[val_idx])\\n        for i, config in enumerate(configs):\\n            model = make_model(config, y[fit_idx])\\n            model.fit(X_fit, y[fit_idx])\\n            pred = model.predict(X_val)\\n            oof[i, val_idx] = pred\\n            score = f1_score(y[val_idx], pred, labels=LABELS, average=\\"macro\\", zero_division=0)\\n            fold_scores[i].append(float(score))\\n            print(f\\"  {config}: F1-macro={score:.5f}\\", flush=True)\\n        del X_fit, X_val, features\\n    cv_scores = [float(f1_score(y, p, labels=LABELS, average=\\"macro\\", zero_division=0)) for p in oof]\\n    best_idx = int(np.argmax(cv_scores))\\n    best = configs[best_idx]\\n    print(f\\"Chọn bằng OOF train: {best}, F1-macro={cv_scores[best_idx]:.5f}\\", flush=True)\\n    print(\\"Huấn luyện cấu hình đã chọn trên toàn bộ train...\\", flush=True)\\n    features = make_features()\\n    X_train = features.fit_transform(texts)\\n    model = make_model(best, y)\\n    model.fit(X_train, y)\\n    dev_pred = model.predict(features.transform(dev_texts))\\n    dev_y = dev[\\"spam_label\\"].to_numpy()\\n    report = metrics(dev_y, dev_pred)\\n    unique_dev = ~pd.Series(dev_texts).isin(set(texts)).to_numpy()\\n    print(f\\"Dev F1-macro={report[\'macro_f1\']:.5f}, accuracy={report[\'accuracy\']:.5f}\\", flush=True)\\n    print(classification_report(dev_y, dev_pred, labels=LABELS, digits=4, zero_division=0), flush=True)\\n    print(f\\"Dev có {int((~unique_dev).sum())} comment chuẩn hóa xuất hiện trong train.\\", flush=True)\\n    test_pred = model.predict(features.transform(test_texts)).astype(int)\\n    by_id = pd.Series(test_pred, index=test[\\"id\\"].astype(str))\\n    submission = data[\\"samplesubmission\\"].copy()\\n    submission[\\"spam_label\\"] = submission[\\"ID\\"].astype(str).map(by_id)\\n    if submission[\\"spam_label\\"].isna().any() or not submission[\\"spam_label\\"].isin(LABELS).all():\\n        raise ValueError(\\"Submission có nhãn thiếu hoặc không hợp lệ\\")\\n    submission[\\"spam_label\\"] = submission[\\"spam_label\\"].astype(int)\\n    submission_path = output / \\"submission.csv\\"\\n    submission.to_csv(submission_path, index=False)\\n    saved = pd.read_csv(submission_path)\\n    pd.testing.assert_frame_equal(saved, submission, check_dtype=False)\\n    if len(saved) != len(test) or not saved[\\"ID\\"].equals(data[\\"samplesubmission\\"][\\"ID\\"]):\\n        raise ValueError(\\"Sai số hàng hoặc thứ tự ID khi lưu\\")\\n    detail = pd.DataFrame({\\"comment\\": dev[\\"comment\\"], \\"true_label\\": dev_y, \\"predicted_label\\": dev_pred,\\n                           \\"comment_seen_in_train\\": ~unique_dev})\\n    detail.to_csv(output / \\"dev_predictions.csv\\", index=False)\\n    full_report = {\\n        \\"seed\\": SEED, \\"python\\": platform.python_version(), \\"sklearn\\": sklearn.__version__,\\n        \\"rows\\": {k: len(v) for k, v in data.items()}, \\"feature_columns\\": [\\"comment\\"],\\n        \\"selection\\": \\"StratifiedGroupKFold on train; normalized comments as groups; OOF macro F1\\",\\n        \\"fit_data\\": \\"train only; dev and test never used to fit TF-IDF, scaler or SVM\\",\\n        \\"candidates\\": [{**c, \\"fold_macro_f1\\": f, \\"oof_macro_f1\\": s}\\n                       for c, f, s in zip(configs, fold_scores, cv_scores)],\\n        \\"chosen_config\\": best, \\"dev\\": report,\\n        \\"dev_comments_seen_in_train\\": int((~unique_dev).sum()),\\n        \\"dev_without_comment_overlap\\": metrics(dev_y[unique_dev], dev_pred[unique_dev]) if unique_dev.any() else None,\\n        \\"test_prediction_counts\\": {int(k): int(v) for k, v in submission.spam_label.value_counts().items()},\\n        \\"submission_rows_verified\\": len(saved),\\n    }\\n    (output / \\"metrics.json\\").write_text(json.dumps(full_report, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    print(f\\"Đã kiểm tra và lưu {len(saved)} dự đoán: {submission_path}\\", flush=True)\\n    return full_report\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "improved_features.py": "\\"\\"\\"Đặc trưng v2: văn bản review và mức trùng với thông tin sản phẩm.\\"\\"\\"\\nimport re\\nfrom collections import Counter\\nimport numpy as np\\nfrom scipy import sparse\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.preprocessing import StandardScaler, OneHotEncoder\\nfrom train_submit import normalize_text, ReviewStats\\n\\n\\ndef prepare(frame):\\n    frame = frame.copy()\\n    for col in [\'comment\', \'product_name\', \'description\', \'category\', \'categories\']:\\n        frame[col] = frame[col].fillna(\'\').astype(str).map(normalize_text)\\n    return frame\\n\\n\\ndef tokens(text):\\n    return re.findall(r\'(?u)\\\\b\\\\w+\\\\b\', text)\\n\\n\\ndef ngrams(words, n):\\n    return set(zip(*(words[i:] for i in range(n))))\\n\\n\\ndef dense_features(df):\\n    basic = ReviewStats().transform(df.comment).toarray()\\n    extra = []\\n    for row in df.itertuples(index=False):\\n        text = row.comment\\n        words = tokens(text)\\n        wl = [len(w) for w in words] or [0]\\n        wc = Counter(words)\\n        n = max(len(words), 1)\\n        feats = [np.log1p(max(wl)), np.mean(wl), np.std(wl),\\n                 sum(len(w) > 12 for w in words) / n,\\n                 max(wc.values(), default=0) / n,\\n                 sum(v * (v - 1) for v in wc.values()) / max(n * (n - 1), 1),\\n                 text.count(\'.\') / max(len(text), 1),\\n                 len(re.findall(r\'(.)\\\\1{3,}\', text)) / n,\\n                 np.log1p(len(text.split())),\\n                 float(row.rating) / 5,\\n                 np.log1p(max(float(row.num_sold), 0)),\\n                 np.log1p(max(float(row.num_review), 0))]\\n        for context in [row.product_name, row.description]:\\n            context_words = tokens(context)\\n            for size in [1, 2, 3, 4]:\\n                a, b = ngrams(words, size), ngrams(context_words, size)\\n                overlap = len(a & b)\\n                ratio = overlap / max(len(a), 1)\\n                feats += [ratio, ratio ** 2, ratio * np.log1p(len(words)), np.log1p(overlap)]\\n            feats += [float(len(text) > 10 and text in context), np.log1p(len(context))]\\n        extra.append(feats)\\n    return np.hstack([basic, np.asarray(extra, dtype=np.float32)])\\n\\n\\nclass FeaturesV2:\\n    def __init__(self):\\n        self.word = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=140000,\\n                                   sublinear_tf=True, lowercase=False, dtype=np.float32,\\n                                   token_pattern=r\'(?u)\\\\b\\\\w+\\\\b\')\\n        self.char = TfidfVectorizer(analyzer=\'char\', ngram_range=(2, 5), min_df=3,\\n                                   max_features=180000, sublinear_tf=True,\\n                                   lowercase=False, dtype=np.float32)\\n        self.product = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20000,\\n                                      sublinear_tf=True, lowercase=False, dtype=np.float32)\\n        self.category = OneHotEncoder(handle_unknown=\'ignore\', dtype=np.float32)\\n        self.scale = StandardScaler()\\n\\n    def fit_transform(self, df):\\n        text = sparse.hstack([self.word.fit_transform(df.comment), self.char.fit_transform(df.comment)], format=\'csr\')\\n        meta = sparse.hstack([self.product.fit_transform(df.product_name) * .2,\\n                             self.category.fit_transform(df[[\'category\', \'rating\']].astype(str)) * .2,\\n                             sparse.csr_matrix(self.scale.fit_transform(dense_features(df))) * .15], format=\'csr\')\\n        return text, meta\\n\\n    def transform(self, df):\\n        text = sparse.hstack([self.word.transform(df.comment), self.char.transform(df.comment)], format=\'csr\')\\n        meta = sparse.hstack([self.product.transform(df.product_name) * .2,\\n                             self.category.transform(df[[\'category\', \'rating\']].astype(str)) * .2,\\n                             sparse.csr_matrix(self.scale.transform(dense_features(df))) * .15], format=\'csr\')\\n        return text, meta\\n\\n\\ndef nb_ratios(X, y):\\n    \\"\\"\\"Log-count ratio one-vs-rest, fit chỉ từ phần train của fold.\\"\\"\\"\\n    ratios = []\\n    for c in range(4):\\n        pos = np.asarray(X[y == c].sum(axis=0)).ravel() + 1\\n        neg = np.asarray(X[y != c].sum(axis=0)).ravel() + 1\\n        ratios.append(np.log((pos / pos.sum()) / (neg / neg.sum())))\\n    return ratios\\n", "select_v2.py": "\\"\\"\\"Chọn mô hình/ensemble và bias bằng OOF train, không xem nhãn dev.\\"\\"\\"\\nimport json\\nfrom pathlib import Path\\nimport numpy as np\\n\\n\\ndef macro(y, pred):\\n    cm = np.bincount(4 * y + pred, minlength=16).reshape(4, 4)\\n    den = cm.sum(axis=0) + cm.sum(axis=1)\\n    return float(np.mean(np.divide(2 * np.diag(cm), den, out=np.zeros(4), where=den > 0)))\\n\\n\\ndef calibrate(scores, y):\\n    # 3 offsets tương đối so với nhãn 0; grid nhỏ để hạn chế overfit.\\n    bias = np.zeros(4)\\n    for _ in range(3):\\n        previous = bias.copy()\\n        for cls in [1, 2, 3]:\\n            choices = []\\n            for value in np.linspace(-.4, .4, 17):\\n                proposal = bias.copy()\\n                proposal[cls] = value\\n                objective = macro(y, (scores + proposal).argmax(axis=1)) - .005 * np.square(proposal).sum()\\n                choices.append((objective, -abs(value), float(value)))\\n            bias[cls] = max(choices)[2]\\n        if np.allclose(previous, bias):\\n            break\\n    return bias\\n\\n\\ndef select(scores, y, folds, configs):\\n    # NB-SVM C=1/2 có cảnh báo chưa hội tụ ở thử nghiệm, không chọn chúng.\\n    candidates = [(cfg[\'name\'], {i: 1.}) for i, cfg in enumerate(configs)\\n                  if cfg[\'kind\'] != \'nb\' or cfg[\'C\'] == .5]\\n    # Blends định trước, tránh tìm hàng trăm trọng số trên lớp hiếm.\\n    byname = {c[\'name\']: i for i, c in enumerate(configs)}\\n    for svm_w in [.25, .5, .75]:\\n        candidates.append((f\'blend_svm_nb_{svm_w}\',\\n                           {byname[\'svm_0.5_balanced\']: svm_w, byname[\'nb_0.5\']: 1-svm_w}))\\n    rows = []\\n    for name, weights in candidates:\\n        s = sum(scores[i] * w for i, w in weights.items())\\n        cross_pred = np.zeros(len(y), dtype=int)\\n        for f in np.unique(folds):\\n            fit = folds != f\\n            b = calibrate(s[fit], y[fit])\\n            cross_pred[~fit] = (s[~fit] + b).argmax(axis=1)\\n        raw, calibrated = macro(y, s.argmax(axis=1)), macro(y, cross_pred)\\n        # Chọn bias chỉ nếu calibration cải thiện khi kiểm tra chéo.\\n        use_bias = calibrated > raw + .002\\n        row = {\'name\': name, \'weights\': weights, \'raw_oof_macro_f1\': raw,\\n               \'cross_calibration_macro_f1\': calibrated, \'use_bias\': use_bias,\\n               \'selection_score\': calibrated if use_bias else raw,\\n               \'bias\': calibrate(s, y).tolist() if use_bias else [0., 0., 0., 0.]}\\n        rows.append(row)\\n    rows.sort(key=lambda r: r[\'selection_score\'], reverse=True)\\n    return rows\\n\\n\\nif __name__ == \'__main__\':\\n    folder = Path(\'kaggle_ds102/experiments_v2\')\\n    z = np.load(folder/\'oof.npz\')\\n    cfg = json.loads((folder/\'configs.json\').read_text())\\n    rows = select(z[\'scores\'], z[\'y\'], z[\'folds\'], cfg)\\n    (folder/\'selection.json\').write_text(json.dumps(rows, indent=2))\\n    for r in rows:\\n        print(r[\'name\'], \'raw\', round(r[\'raw_oof_macro_f1\'], 5),\\n              \'cross calibrated\', round(r[\'cross_calibration_macro_f1\'], 5), \'bias\', r[\'bias\'])\\n", "train_submit_v2.py": "\\"\\"\\"DS102 v2: đặc trưng quan hệ review/sản phẩm, SVM và calibration F1-macro.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport platform\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nimport sklearn\\nfrom scipy import sparse\\nfrom sklearn.metrics import classification_report\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom sklearn.svm import LinearSVC\\nfrom train_submit import load_data, find_data_dir, make_model, make_features, metrics, LABELS, SEED\\nfrom improved_features import prepare, FeaturesV2, nb_ratios\\nfrom select_v2 import macro, calibrate\\n\\n\\n# Cấu hình được chốt từ cross-validation trên train, trước đánh giá dev v2.\\nRECIPE = [\\n    {\'kind\': \'svm\', \'C\': .5, \'weight\': \'balanced\', \'ensemble_weight\': .75},\\n    {\'kind\': \'nb\', \'C\': .5, \'weight\': \'balanced\', \'ensemble_weight\': .25},\\n]\\n\\n\\ndef fit_predict(fit_df, predict_frames, recipe):\\n    \\"\\"\\"Không nhận nhãn của các tập dự đoán.\\"\\"\\"\\n    y = fit_df.spam_label.to_numpy()\\n    features = FeaturesV2()\\n    Xt, Mt = features.fit_transform(fit_df)\\n    transformed = [features.transform(df) for df in predict_frames]\\n    result = [np.zeros((len(df), 4)) for df in predict_frames]\\n    ratios = nb_ratios(Xt, y) if any(c[\'kind\'] == \'nb\' for c in recipe) else None\\n    for config in recipe:\\n        if config[\'kind\'] == \'svm\':\\n            model = make_model(config, y).fit(sparse.hstack([Xt, Mt], format=\'csr\'), y)\\n            if np.max(model.n_iter_) >= model.max_iter:\\n                raise RuntimeError(\'SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.\')\\n            for j, (Xv, Mv) in enumerate(transformed):\\n                result[j] += config[\'ensemble_weight\'] * model.decision_function(sparse.hstack([Xv, Mv], format=\'csr\'))\\n        elif config[\'kind\'] == \'nb\':\\n            for cls, r in enumerate(ratios):\\n                model = LinearSVC(C=config[\'C\'], class_weight=\'balanced\', dual=True,\\n                                  max_iter=30000, random_state=SEED)\\n                model.fit(sparse.hstack([Xt.multiply(r), Mt], format=\'csr\'), (y == cls).astype(int))\\n                if np.max(model.n_iter_) >= model.max_iter:\\n                    raise RuntimeError(\'NB-SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.\')\\n                for j, (Xv, Mv) in enumerate(transformed):\\n                    result[j][:, cls] += config[\'ensemble_weight\'] * model.decision_function(\\n                        sparse.hstack([Xv.multiply(r), Mv], format=\'csr\'))\\n        else:\\n            raise ValueError(f\\"Model không được hỗ trợ: {config[\'kind\']}\\")\\n    return result\\n\\n\\ndef main_v2(argv=None):\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--data-dir\', default=None)\\n    parser.add_argument(\'--output-dir\', default=\'output_v2\')\\n    parser.add_argument(\'--folds\', type=int, default=3)\\n    args = parser.parse_args(argv)\\n    if args.folds < 2:\\n        parser.error(\'--folds phải >= 2\')\\n    data_dir = find_data_dir(args.data_dir)\\n    data = load_data(data_dir)\\n    expected = {\'comment\', \'product_name\', \'description\', \'category\', \'categories\', \'rating\', \'num_sold\', \'num_review\'}\\n    for name in [\'train\', \'dev\', \'test\']:\\n        missing = expected - set(data[name].columns)\\n        if missing:\\n            raise ValueError(f\'{name} thiếu cột {missing}\')\\n        for col in [\'rating\', \'num_sold\', \'num_review\']:\\n            values = pd.to_numeric(data[name][col], errors=\'raise\')\\n            if not np.isfinite(values).all() or (values < 0).any():\\n                raise ValueError(f\'{name}.{col}: giá trị số không hợp lệ\')\\n            data[name][col] = values\\n    train, dev, test = [prepare(data[n]) for n in [\'train\', \'dev\', \'test\']]\\n    output = Path(args.output_dir).expanduser().resolve()\\n    output.mkdir(parents=True, exist_ok=True)\\n    y = train.spam_label.to_numpy()\\n    oof = np.zeros((len(train), 4))\\n    fold_ids = np.full(len(train), -1)\\n    splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=SEED)\\n    for fold, (tr, va) in enumerate(splitter.split(train, y, groups=train.comment)):\\n        if set(y[tr]) != set(LABELS):\\n            raise ValueError(\'Fold không đủ 4 lớp; giảm số folds.\')\\n        print(f\'CV fold {fold+1}/{args.folds}\', flush=True)\\n        oof[va] = fit_predict(train.iloc[tr], [train.iloc[va].drop(columns=\'spam_label\')], RECIPE)[0]\\n        fold_ids[va] = fold\\n        print(f\'  Raw F1-macro={macro(y[va], oof[va].argmax(axis=1)):.5f}\', flush=True)\\n    cross_pred = np.zeros(len(train), dtype=int)\\n    for f in range(args.folds):\\n        fit = fold_ids != f\\n        bias_fold = calibrate(oof[fit], y[fit])\\n        cross_pred[~fit] = (oof[~fit] + bias_fold).argmax(axis=1)\\n    raw_score = macro(y, oof.argmax(axis=1))\\n    cross_score = macro(y, cross_pred)\\n    use_bias = cross_score > raw_score + .002\\n    bias = calibrate(oof, y) if use_bias else np.zeros(4)\\n    print(f\'OOF raw={raw_score:.5f}; cross-calibrated={cross_score:.5f}; bias={bias.tolist()}\', flush=True)\\n    print(\'Fit toàn bộ train và dự đoán dev/test...\', flush=True)\\n    dev_scores, test_scores = fit_predict(train, [dev.drop(columns=\'spam_label\'), test], RECIPE)\\n    dev_pred = (dev_scores + bias).argmax(axis=1)\\n    test_pred = (test_scores + bias).argmax(axis=1)\\n    dev_y = dev.spam_label.to_numpy()\\n    unique_dev = ~dev.comment.isin(set(train.comment)).to_numpy()\\n    dev_report = metrics(dev_y, dev_pred)\\n    print(f\\"Dev F1-macro={dev_report[\'macro_f1\']:.5f}\\", flush=True)\\n    print(classification_report(dev_y, dev_pred, labels=LABELS, digits=4, zero_division=0), flush=True)\\n    by_id = pd.Series(test_pred, index=test.id.astype(str))\\n    submission = data[\'samplesubmission\'].copy()\\n    submission[\'spam_label\'] = submission.ID.astype(str).map(by_id)\\n    if submission.spam_label.isna().any() or not submission.spam_label.isin(LABELS).all():\\n        raise ValueError(\'Nhãn submission không hợp lệ\')\\n    submission[\'spam_label\'] = submission.spam_label.astype(int)\\n    path = output/\'submission_v2.csv\'\\n    submission.to_csv(path, index=False)\\n    saved = pd.read_csv(path)\\n    pd.testing.assert_frame_equal(saved, submission, check_dtype=False)\\n    if len(saved) != len(test) or not saved.ID.equals(data[\'samplesubmission\'].ID):\\n        raise ValueError(\'Sai số dòng hoặc thứ tự ID\')\\n    pd.DataFrame({\'comment\': data[\'dev\'].comment, \'true_label\': dev_y,\\n                  \'predicted_label\': dev_pred, \'comment_seen_in_train\': ~unique_dev}).to_csv(output/\'dev_predictions_v2.csv\', index=False)\\n    np.savez_compressed(output/\'scores_v2.npz\', oof=oof, y=y, folds=fold_ids,\\n                        dev_scores=dev_scores, test_scores=test_scores, bias=bias)\\n    report = {\\n        \'recipe\': RECIPE, \'seed\': SEED, \'folds\': args.folds, \'python\': platform.python_version(),\\n        \'sklearn\': sklearn.__version__, \'data_sha256\': {name: hashlib.sha256((data_dir/file).read_bytes()).hexdigest()\\n            for name, file in [(\'train\',\'ds102_train.csv\'), (\'dev\',\'ds102_dev.csv\'), (\'test\',\'ds102_test.csv\')]},\\n        \'fit_data\': \'train only; no dev or test used to fit features, models, or calibration\',\\n        \'oof_raw_macro_f1\': raw_score, \'oof_cross_calibration_macro_f1\': cross_score,\\n        \'calibration_bias\': bias.tolist(), \'dev\': dev_report,\\n        \'dev_without_comment_overlap\': metrics(dev_y[unique_dev], dev_pred[unique_dev]) if unique_dev.any() else None,\\n        \'selection_caveat\': \'CV estimates were used for model selection; dev was previously used to inspect baseline errors.\',\\n        \'submission_rows_verified\': len(saved),\\n        \'prediction_counts\': {int(k): int(v) for k, v in saved.spam_label.value_counts().items()},\\n    }\\n    (output/\'metrics_v2.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\\n    print(f\'Đã kiểm tra và lưu {len(saved)} dòng: {path}\', flush=True)\\n    return report\\n\\n\\nif __name__ == \'__main__\':\\n    main_v2()\\n", "catboost_v3.py": "\\"\\"\\"CatBoost text + thông tin review/sản phẩm, không dùng mô hình pretrained.\\"\\"\\"\\nimport numpy as np\\nimport pandas as pd\\nfrom catboost import CatBoostClassifier\\nfrom improved_features import dense_features, tokens\\nfrom train_submit import SEED\\n\\n\\ndef cat_features(df):\\n    a = dense_features(df)\\n    x = pd.DataFrame(a, columns=[f\'stat_{i}\' for i in range(a.shape[1])])\\n    x[\'comment\'] = [\' \'.join(tokens(t)) or \'emptyreview\' for t in df.comment]\\n    x[\'category\'] = df.category.astype(str).to_numpy()\\n    x[\'rating\'] = df.rating.astype(str).to_numpy()\\n    return x\\n\\n\\ndef make_cat(weight=\'SqrtBalanced\', iterations=500):\\n    return CatBoostClassifier(\\n        iterations=iterations, depth=5, learning_rate=.1, l2_leaf_reg=5,\\n        loss_function=\'MultiClass\', auto_class_weights=weight,\\n        thread_count=4, random_seed=SEED, allow_writing_files=False,\\n        text_features=[\'comment\'], cat_features=[\'category\', \'rating\'],\\n        tokenizers=[{\'tokenizer_id\':\'Space\', \'separator_type\':\'ByDelimiter\', \'delimiter\':\' \'}],\\n        dictionaries=[{\'dictionary_id\':\'Word\', \'gram_order\':\'1\', \'occurrence_lower_bound\':\'3\', \'max_dictionary_size\':\'40000\'},\\n                      {\'dictionary_id\':\'Bigram\', \'gram_order\':\'2\', \'occurrence_lower_bound\':\'3\', \'max_dictionary_size\':\'40000\'}],\\n        feature_calcers=[\'BoW:top_tokens_count=700\', \'NaiveBayes\', \'BM25\'],\\n        verbose=False,\\n    )\\n", "experiment_v5.py": "\\"\\"\\"Hosted Colab only: compare new sparse text views and richer CatBoost.\\n\\nNo pretrained weights, embeddings, outside data, or development labels for\\nselection. All vocabularies and scales fit inside grouped training folds.\\n\\"\\"\\"\\nimport gc\\nimport json\\nimport os\\nimport platform\\nimport traceback\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy import sparse\\nfrom scipy.special import softmax\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom sklearn.preprocessing import StandardScaler, OneHotEncoder\\nfrom sklearn.svm import LinearSVC\\nfrom improved_features import prepare, dense_features, tokens\\nfrom train_submit import load_data, SEED, metrics\\nfrom select_v2 import macro\\n\\n\\ndef hosted_guard():\\n    if platform.system() != \'Linux\' or not os.environ.get(\'COLAB_RELEASE_TAG\'):\\n        raise RuntimeError(\'Training requires hosted Google Colab; local fitting disabled.\')\\n\\n\\nclass TextViews:\\n    def __init__(self):\\n        self.word = TfidfVectorizer(ngram_range=(1, 3), min_df=2,\\n            max_features=160000, sublinear_tf=True, token_pattern=r\'(?u)\\\\b\\\\w+\\\\b\',\\n            dtype=np.float32)\\n        self.char = TfidfVectorizer(analyzer=\'char_wb\', ngram_range=(3, 6),\\n            min_df=3, max_features=220000, sublinear_tf=True, dtype=np.float32)\\n        self.ascii = TfidfVectorizer(analyzer=\'char_wb\', strip_accents=\'unicode\',\\n            ngram_range=(3, 5), min_df=4, max_features=100000,\\n            sublinear_tf=True, dtype=np.float32)\\n        self.product = TfidfVectorizer(ngram_range=(1, 2), min_df=3,\\n            max_features=25000, sublinear_tf=True, dtype=np.float32)\\n        self.category = OneHotEncoder(handle_unknown=\'ignore\', dtype=np.float32)\\n        self.scaler = StandardScaler()\\n\\n    def build(self, df, fit=False):\\n        def transform(model, values):\\n            return model.fit_transform(values) if fit else model.transform(values)\\n        text = sparse.hstack([transform(self.word, df.comment),\\n            transform(self.char, df.comment), .5*transform(self.ascii, df.comment)], format=\'csr\')\\n        nums = dense_features(df)\\n        nums = self.scaler.fit_transform(nums) if fit else self.scaler.transform(nums)\\n        # Limit rare metadata outliers, retaining previously validated features.\\n        nums = np.clip(nums, -5, 5).astype(np.float32)\\n        meta = sparse.hstack([.3*transform(self.product, df.product_name),\\n            .3*transform(self.category, df[[\'category\', \'rating\']].astype(str)),\\n            .15*sparse.csr_matrix(nums)], format=\'csr\')\\n        return sparse.hstack([text, meta], format=\'csr\')\\n\\n\\nLINEAR_CONFIGS = [dict(C=.5, weight=\'balanced\'), dict(C=1., weight=\'sqrt\'),\\n                  dict(C=2., weight=\'sqrt\')]\\n\\n\\ndef linear_fit_predict(fit, frames, config):\\n    hosted_guard()\\n    f = TextViews(); x = f.build(fit, fit=True)\\n    yy = fit.spam_label.to_numpy()\\n    weight = config[\'weight\']\\n    if weight == \'sqrt\':\\n        counts = np.bincount(yy, minlength=4)\\n        weight = {c:float(np.sqrt(len(yy)/(4*counts[c]))) for c in range(4)}\\n    model = LinearSVC(C=config[\'C\'], class_weight=weight, dual=True,\\n        max_iter=20000, random_state=SEED)\\n    model.fit(x, yy)\\n    if np.max(model.n_iter_) >= model.max_iter:\\n        raise RuntimeError(\'Linear SVM did not converge; do not select this run.\')\\n    return [softmax(3*model.decision_function(f.build(d)), axis=1) for d in frames]\\n\\n\\ndef rich_cat_features(df):\\n    from catboost_v3 import cat_features\\n    x = cat_features(df)\\n    # Product/category identities are ordinary supplied input features, with\\n    # CatBoost\'s ordered target statistics, never external labels.\\n    x[\'product_name\'] = df.product_name.to_numpy()\\n    x[\'categories\'] = df.categories.astype(str).to_numpy()\\n    n = np.maximum(df.num_review.astype(float).to_numpy(), 1)\\n    x[\'sold_per_review\'] = np.log1p(df.num_sold.astype(float).to_numpy()/n)\\n    return x\\n\\n\\ndef make_rich_cat():\\n    from catboost_v3 import make_cat\\n    m = make_cat(\'SqrtBalanced\', iterations=900)\\n    m.set_params(depth=6, learning_rate=.07, l2_leaf_reg=8, thread_count=2,\\n        cat_features=[\'category\', \'rating\', \'product_name\', \'categories\'],\\n        feature_calcers=[\'BoW:top_tokens_count=2000\', \'NaiveBayes\', \'BM25\'])\\n    return m\\n\\n\\ndef main(data_dir=\'/content/ds102_data\', output_dir=\'/content/ds102_v5_experiment\',\\n         family=\'linear\'):\\n    hosted_guard()\\n    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)\\n    def status(**values):\\n        (out/\'status.json\').write_text(json.dumps(values, ensure_ascii=False, indent=2))\\n    try:\\n        df = prepare(load_data(Path(data_dir))[\'train\'])\\n        y = df.spam_label.to_numpy(); folds = np.full(len(y), -1)\\n        configs = LINEAR_CONFIGS if family == \'linear\' else [{\'model\':\'rich_cat\', \'trees\':500}, {\'model\':\'rich_cat\', \'trees\':900}]\\n        oof = np.zeros((len(configs),len(y),4), dtype=np.float32)\\n        results = [[] for _ in configs]\\n        splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)\\n        for fold,(tr,va) in enumerate(splitter.split(df,y,groups=df.comment)):\\n            folds[va] = fold\\n            train, valid = df.iloc[tr], df.iloc[va].drop(columns=\'spam_label\')\\n            if family == \'linear\':\\n                status(stage=\'features\',fold=fold+1,results=results)\\n                f = TextViews(); xt=f.build(train,fit=True); xv=f.build(valid)\\n                for i,c in enumerate(configs):\\n                    status(stage=\'fit\',fold=fold+1,config=c,results=results)\\n                    weight=c[\'weight\']\\n                    if weight==\'sqrt\':\\n                        counts=np.bincount(y[tr],minlength=4)\\n                        weight={k:float(np.sqrt(len(tr)/(4*counts[k]))) for k in range(4)}\\n                    m=LinearSVC(C=c[\'C\'],class_weight=weight,dual=True,\\n                        max_iter=20000,random_state=SEED).fit(xt,y[tr])\\n                    if np.max(m.n_iter_) >= m.max_iter:\\n                        raise RuntimeError(\'Linear SVM failed to converge\')\\n                    oof[i,va]=softmax(3*m.decision_function(xv),axis=1)\\n                    results[i].append(macro(y[va],oof[i,va].argmax(1)))\\n                del f,xt,xv,m\\n            else:\\n                status(stage=\'fit\',fold=fold+1,config=\'rich_cat_900\',results=results)\\n                m=make_rich_cat(); m.fit(rich_cat_features(train),y[tr],verbose=300)\\n                for i,c in enumerate(configs):\\n                    oof[i,va]=m.predict_proba(rich_cat_features(valid),ntree_end=c[\'trees\'])\\n                    results[i].append(macro(y[va],oof[i,va].argmax(1)))\\n                del m\\n            gc.collect()\\n            np.savez_compressed(out/\'partial.npz\',oof=oof,y=y,folds=folds)\\n            status(stage=\'fold_complete\',fold=fold+1,results=results)\\n        report={\'family\':family,\'configs\':configs,\'fold_scores\':results,\\n            \'oof_macro_f1\':[macro(y,p.argmax(1)) for p in oof],\\n            \'fit_data\':\'train only\',\'pretrained\':False,\'execution\':\'hosted Colab CPU\'}\\n        np.savez_compressed(out/\'oof.npz\',oof=oof,y=y,folds=folds)\\n        (out/\'cv_results.json\').write_text(json.dumps(report,indent=2))\\n        status(stage=\'complete\',**report)\\n        return report\\n    except BaseException as exc:\\n        status(stage=\'error\',error=str(exc),traceback=traceback.format_exc())\\n        raise\\n", "selection_v5.py": "\\"\\"\\"Coarse probability reweighting, validated on separate OOF folds.\\"\\"\\"\\nfrom itertools import product\\nimport numpy as np\\nfrom select_v2 import macro\\n\\n\\ndef fit_weights(probabilities, y):\\n    choices = []\\n    # A small fixed grid: minority-class recall matters for macro F1.\\n    # Identity is included. Never access development or test labels here.\\n    for a,b,c in product([.8,1.,1.2,1.4], [.8,1.,1.2,1.4], [.9,1.,1.1]):\\n        w=np.array([1.,a,b,c])\\n        score=macro(y,(probabilities*w).argmax(1))\\n        penalty=.002*float(np.square(np.log(w)).sum())\\n        choices.append((score-penalty,-float(np.square(np.log(w)).sum()),w))\\n    return max(choices,key=lambda row:row[:2])[2]\\n\\n\\ndef cross_weight(probabilities,y,folds):\\n    pred=np.empty(len(y),dtype=int); weights=[]; fold_deltas=[]\\n    for fold in np.unique(folds):\\n        fit=folds!=fold\\n        w=fit_weights(probabilities[fit],y[fit]); weights.append(w.tolist())\\n        pred[~fit]=(probabilities[~fit]*w).argmax(1)\\n        fold_deltas.append(macro(y[~fit],pred[~fit])-\\n            macro(y[~fit],probabilities[~fit].argmax(1)))\\n    raw=macro(y,probabilities.argmax(1)); cross=macro(y,pred)\\n    use=bool(cross>=raw+.002 and min(fold_deltas)>=-.002)\\n    return {\'raw_macro_f1\':raw,\'cross_weight_macro_f1\':cross,\\n        \'fold_deltas\':fold_deltas,\'fold_weights\':weights,\'use_weights\':use,\\n        \'weights\':fit_weights(probabilities,y).tolist() if use else [1.,1.,1.,1.],\\n        \'selection_score\':cross if use else raw}\\n\\n\\ndef compare(reference,new_oof,y,folds,configs):\\n    rows=[]\\n    row=cross_weight(reference,y,folds)\\n    row.update(model_index=-1,config=\'v3_reference\',new_model_weight=0.)\\n    rows.append(row)\\n    for i,values in enumerate(new_oof):\\n        for alpha in [.15,.3,.5,1.]:\\n            p=(1-alpha)*reference+alpha*values\\n            row=cross_weight(p,y,folds)\\n            row.update(model_index=i,config=configs[i],new_model_weight=alpha)\\n            rows.append(row)\\n    return sorted(rows,key=lambda row:row[\'selection_score\'],reverse=True)\\n", "temperature_v5.py": "\\"\\"\\"Select coarse SVM temperature / CatBoost mixing using saved train OOF.\\"\\"\\"\\nimport json\\nfrom pathlib import Path\\nimport numpy as np\\nfrom scipy.special import softmax\\nfrom experiment_v5 import hosted_guard\\nfrom selection_v5 import cross_weight\\nfrom select_v2 import macro\\n\\n\\ndef main(component_path=\'/content/colab_components_v5.npz\',\\n         output_dir=\'/content/ds102_v5_temperature\'):\\n    hosted_guard()\\n    z=np.load(component_path);y=z[\'y\'];folds=z[\'folds\']\\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\\n    rows=[]\\n    # Fixed coarse candidates. Original v3 (temperature=3, cat_weight=.5)\\n    # remains in the grid. No development labels are read by this function.\\n    for temperature in [1.5,3.,6.]:\\n        for cat_weight in [.25,.5,.75]:\\n            p=(1-cat_weight)*softmax(temperature*z[\'svm_oof\'],axis=1)+cat_weight*z[\'cat_oof\']\\n            row=cross_weight(p,y,folds)\\n            row.update(config={\'temperature\':temperature,\'cat_weight\':cat_weight},\\n                       family=\'temperature\',new_model_weight=1.,model_index=0)\\n            row[\'raw_fold_f1\']=[macro(y[folds==f],p[folds==f].argmax(1)) for f in np.unique(folds)]\\n            rows.append(row)\\n    rows.sort(key=lambda r:r[\'selection_score\'],reverse=True)\\n    (out/\'selection.json\').write_text(json.dumps(rows,indent=2))\\n    (out/\'status.json\').write_text(json.dumps({\'stage\':\'complete\',\'best\':rows[0]},indent=2))\\n    return rows\\n", "fasttext_scratch.py": "\\"\\"\\"Optional fastText classifier trained from random initialization on Colab.\\n\\nUses the official MIT-licensed v0.9.2 C++ CLI. No pretrained vectors, downloaded\\nmodel, external corpus, or Python fastText bindings are used.\\n\\"\\"\\"\\nimport gc\\nimport hashlib\\nimport json\\nimport math\\nimport subprocess\\nimport traceback\\nimport urllib.request\\nimport zipfile\\nfrom pathlib import Path\\nimport numpy as np\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom experiment_v5 import hosted_guard\\nfrom improved_features import prepare,tokens\\nfrom train_submit import load_data,SEED\\nfrom select_v2 import macro\\n\\nCONFIGS=[{\'balance_power\':.5},{\'balance_power\':.75}]\\n\\n\\ndef install():\\n    hosted_guard()\\n    root=Path(\'/content/fastText-0.9.2\'); binary=root/\'fasttext\'\\n    if not binary.exists():\\n        archive=Path(\'/content/fastText-0.9.2.zip\')\\n        urllib.request.urlretrieve(\'https://github.com/facebookresearch/fastText/archive/refs/tags/v0.9.2.zip\',archive)\\n        with zipfile.ZipFile(archive) as z: z.extractall(\'/content\')\\n        # GCC in recent Colab images requires an explicit cstdint include for\\n        # the unmodified 0.9.2 source. This changes compilation, not the model.\\n        subprocess.run([\'make\',\'-j2\',\'CXX=c++ -include cstdint\'],cwd=root,\\n                       check=True,capture_output=True,text=True)\\n        (root/\'archive_sha256.txt\').write_text(hashlib.sha256(archive.read_bytes()).hexdigest())\\n    return binary\\n\\n\\ndef document(row):\\n    words=[t for t in tokens(row.comment) if not t.startswith(\'__label__\')]\\n    title=set(tokens(row.product_name)); description=set(tokens(row.description))\\n    s=set(words); n=max(len(s),1)\\n    context=[f\'rating_{row.rating}\',f\'lengthbin_{int(math.log2(max(len(words),1)))}\',\\n             f\'titleoverlap_{int(8*len(s&title)/n)}\',\\n             f\'descoverlap_{int(8*len(s&description)/n)}\']\\n    context += [\'product_\'+w for w in tokens(row.product_name)[:30]]\\n    context += [\'category_\'+w for w in tokens(str(row.category))[:10]]\\n    return \' \'.join(words[:300]+context)\\n\\n\\ndef fit_predict(train,frames,config,folder):\\n    hosted_guard()\\n    binary=install();folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\\n    y=train.spam_label.to_numpy();counts=np.bincount(y,minlength=4)\\n    rng=np.random.default_rng(SEED)\\n    lines=[]\\n    for row in train.itertuples(index=False):\\n        factor=(counts.max()/counts[int(row.spam_label)])**config[\'balance_power\']\\n        copies=int(factor)+int(rng.random()<factor-int(factor))\\n        lines += [f\'__label__{int(row.spam_label)} \'+document(row)]*copies\\n    rng.shuffle(lines)\\n    corpus=folder/\'train.txt\';corpus.write_text(\'\\\\n\'.join(lines)+\'\\\\n\')\\n    prefix=folder/\'model\'\\n    command=[str(binary),\'supervised\',\'-input\',str(corpus),\'-output\',str(prefix),\\n             \'-lr\',\'0.5\',\'-epoch\',\'25\',\'-dim\',\'64\',\'-wordNgrams\',\'2\',\\n             \'-minn\',\'2\',\'-maxn\',\'5\',\'-bucket\',\'100000\',\'-minCount\',\'2\',\\n             \'-loss\',\'softmax\',\'-thread\',\'1\',\'-verbose\',\'0\']\\n    subprocess.run(command,check=True,capture_output=True,text=True)\\n    result=[]\\n    for i,df in enumerate(frames):\\n        path=folder/f\'predict_{i}.txt\'\\n        path.write_text(\'\\\\n\'.join(document(row) for row in df.itertuples(index=False))+\'\\\\n\')\\n        output=subprocess.run([str(binary),\'predict-prob\',str(prefix)+\'.bin\',str(path),\'4\'],\\n                              check=True,capture_output=True,text=True).stdout.splitlines()\\n        if len(output)!=len(df): raise RuntimeError(\'fastText prediction count mismatch\')\\n        p=np.zeros((len(df),4),dtype=np.float32)\\n        for k,line in enumerate(output):\\n            values=line.split()\\n            for label,probability in zip(values[::2],values[1::2]):\\n                p[k,int(label.removeprefix(\'__label__\'))]=float(probability)\\n        p/=p.sum(axis=1,keepdims=True)\\n        if not np.isfinite(p).all(): raise RuntimeError(\'Invalid fastText probabilities\')\\n        result.append(p)\\n    return result\\n\\n\\ndef main(data_dir=\'/content/ds102_data\',output_dir=\'/content/ds102_v5_fasttext\'):\\n    hosted_guard()\\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\\n    def status(**values):\\n        (out/\'status.json\').write_text(json.dumps(values,indent=2))\\n    try:\\n        status(stage=\'install_official_source\');install()\\n        df=prepare(load_data(Path(data_dir))[\'train\']);y=df.spam_label.to_numpy()\\n        folds=np.full(len(y),-1);oof=np.zeros((len(CONFIGS),len(y),4),dtype=np.float32)\\n        results=[[] for _ in CONFIGS]\\n        split=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)\\n        for f,(tr,va) in enumerate(split.split(df,y,groups=df.comment)):\\n            folds[va]=f\\n            for i,c in enumerate(CONFIGS):\\n                status(stage=\'fit\',fold=f+1,config=c,results=results)\\n                oof[i,va]=fit_predict(df.iloc[tr],[df.iloc[va].drop(columns=\'spam_label\')],c,out/f\'fold{f}_config{i}\')[0]\\n                results[i].append(macro(y[va],oof[i,va].argmax(1)))\\n                gc.collect()\\n            np.savez_compressed(out/\'partial.npz\',oof=oof,y=y,folds=folds)\\n        report={\'family\':\'fasttext\',\'configs\':CONFIGS,\'fold_scores\':results,\\n                \'oof_macro_f1\':[macro(y,p.argmax(1)) for p in oof],\\n                \'pretrained\':False,\'license\':\'MIT\',\'fit_data\':\'train only\',\\n                \'initialization\':\'random; no pretrainedVectors argument\'}\\n        np.savez_compressed(out/\'oof.npz\',oof=oof,y=y,folds=folds)\\n        (out/\'cv_results.json\').write_text(json.dumps(report,indent=2));status(stage=\'complete\',**report)\\n        return report\\n    except BaseException as exc:\\n        status(stage=\'error\',error=str(exc),traceback=traceback.format_exc())\\n        raise\\n", "stacking_v5.py": "\\"\\"\\"Regularized stacking of training OOF SVM margins and CatBoost probabilities.\\n\\nThe meta validation reuses base OOF predictions; it is not fully nested CV.\\nDevelopment remains separate and is inspected only after configuration choice.\\n\\"\\"\\"\\nimport gc,json\\nfrom pathlib import Path\\nimport numpy as np\\nfrom scipy.special import softmax\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.pipeline import make_pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom experiment_v5 import hosted_guard\\nfrom train_submit import SEED\\nfrom select_v2 import macro\\n\\n\\nCONFIGS=[{\'C\':c,\'weight\':w} for c in [.1,1.] for w in [\'none\',\'sqrt\']]\\n\\n\\ndef features(margins,cat):\\n    return np.hstack([margins,np.log(np.clip(cat,1e-6,1.))])\\n\\n\\ndef fit_meta(x,y,config):\\n    hosted_guard()\\n    weights=None\\n    if config[\'weight\']==\'sqrt\':\\n        counts=np.bincount(y,minlength=4)\\n        weights={c:float(np.sqrt(len(y)/(4*counts[c]))) for c in range(4)}\\n    m=make_pipeline(StandardScaler(),LogisticRegression(C=config[\'C\'],\\n        class_weight=weights,solver=\'lbfgs\',max_iter=1000,random_state=SEED))\\n    m.fit(x,y)\\n    if np.max(m[-1].n_iter_)>=m[-1].max_iter:raise RuntimeError(\'Meta model did not converge\')\\n    return m\\n\\n\\ndef make_base_oof(train):\\n    \\"\\"\\"For a fully reproducible fresh fit, regenerate level-one training OOF.\\"\\"\\"\\n    hosted_guard()\\n    from finalize_v5 import baseline_predict\\n    y=train.spam_label.to_numpy();svm=np.zeros((len(y),4));cat=np.zeros_like(svm)\\n    split=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)\\n    for fold,(tr,va) in enumerate(split.split(train,y,groups=train.comment)):\\n        print(\'Regenerate base OOF:\',fold+1,flush=True)\\n        s,p=baseline_predict(train.iloc[tr],[train.iloc[va].drop(columns=\'spam_label\')],return_components=True)\\n        svm[va]=s[0];cat[va]=p[0];gc.collect()\\n    return svm,cat\\n\\n\\ndef main(component_path=\'/content/colab_components_v5.npz\',output_dir=\'/content/ds102_v5_stacking\'):\\n    hosted_guard()\\n    z=np.load(component_path);y=z[\'y\'];folds=z[\'folds\']\\n    x=features(z[\'svm_oof\'],z[\'cat_oof\'])\\n    base=.5*softmax(3*z[\'svm_oof\'],axis=1)+.5*z[\'cat_oof\']\\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\\n    oof=np.zeros((len(CONFIGS),len(y),4),dtype=np.float32);rows=[]\\n    for i,cfg in enumerate(CONFIGS):\\n        for fold in np.unique(folds):\\n            fit=folds!=fold;m=fit_meta(x[fit],y[fit],cfg)\\n            oof[i,~fit]=m.predict_proba(x[~fit])\\n        for alpha in [.5,1.]:\\n            p=(1-alpha)*base+alpha*oof[i]\\n            fold_scores=[macro(y[folds==f],p[folds==f].argmax(1)) for f in np.unique(folds)]\\n            rows.append({\'family\':\'stacking\',\'config\':cfg,\'model_index\':i,\\n                \'new_model_weight\':alpha,\'weights\':[1.,1.,1.,1.],\\n                \'raw_macro_f1\':macro(y,p.argmax(1)),\\n                \'selection_score\':macro(y,p.argmax(1)),\\n                \'fold_macro_f1\':fold_scores,\\n                \'caveat\':\'Meta CV on reused base OOF, not fully nested; dev must confirm.\'})\\n    rows.sort(key=lambda r:r[\'selection_score\'],reverse=True)\\n    np.savez_compressed(out/\'oof.npz\',oof=oof,y=y,folds=folds)\\n    (out/\'selection.json\').write_text(json.dumps(rows,indent=2))\\n    (out/\'status.json\').write_text(json.dumps({\'stage\':\'complete\',\'best\':rows[0]},indent=2))\\n    return rows\\n", "finalize_v5.py": "\\"\\"\\"Fit a configuration selected on train OOF; evaluate dev only afterwards.\\"\\"\\"\\nimport gc\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy.special import softmax\\nfrom experiment_v5 import hosted_guard,linear_fit_predict,make_rich_cat,rich_cat_features\\nfrom improved_features import prepare\\nfrom train_submit import load_data,metrics\\n\\n\\ndef baseline_predict(train,frames,return_components=False):\\n    from train_submit_v2 import fit_predict,RECIPE\\n    from catboost_v3 import make_cat,cat_features\\n    margins=fit_predict(train,frames,RECIPE)\\n    gc.collect()\\n    m=make_cat(\'SqrtBalanced\',iterations=500);m.set_params(thread_count=2)\\n    m.fit(cat_features(train),train.spam_label.to_numpy(),verbose=250)\\n    cat=[m.predict_proba(cat_features(d)) for d in frames]\\n    if return_components:return margins,cat\\n    return [.5*softmax(3*s,axis=1)+.5*p for s,p in zip(margins,cat)]\\n\\n\\ndef main(selection,family,data_dir=\'/content/ds102_data\',\\n         output_dir=\'/content/ds102_v5\',reference_path=None,components_path=None):\\n    hosted_guard()\\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\\n    def status(stage,**values):\\n        (out/\'status.json\').write_text(json.dumps(dict(stage=stage,**values),indent=2))\\n    try:\\n        data=load_data(Path(data_dir))\\n        train,dev,test=[prepare(data[k]) for k in [\'train\',\'dev\',\'test\']]\\n        frames=[dev.drop(columns=\'spam_label\'),test]\\n        status(\'baseline_predictions\')\\n        if family in [\'temperature\',\'stacking\']:\\n            if components_path:\\n                z=np.load(components_path)\\n                assert np.array_equal(z[\'y\'],train.spam_label.to_numpy())\\n                margins=[z[\'svm_dev\'],z[\'svm_test\']];cat_values=[z[\'cat_dev\'],z[\'cat_test\']]\\n                baseline_source=\'user-authorized cached SVM and CatBoost predictions\'\\n                if family==\'stacking\': meta_svm,meta_cat=z[\'svm_oof\'],z[\'cat_oof\']\\n            else:\\n                if family==\'stacking\':\\n                    from stacking_v5 import make_base_oof\\n                    meta_svm,meta_cat=make_base_oof(train)\\n                margins,cat_values=baseline_predict(train,frames,return_components=True)\\n                baseline_source=\'freshly fitted from train on hosted Colab\'\\n            base=[.5*softmax(3*s,axis=1)+.5*p for s,p in zip(margins,cat_values)]\\n        elif reference_path:\\n            z=np.load(reference_path)\\n            assert np.array_equal(z[\'y\'],train.spam_label.to_numpy())\\n            assert z[\'dev\'].shape==(len(dev),4) and z[\'test\'].shape==(len(test),4)\\n            # This cache was checksum-verified against the previously permitted\\n            # v3 scores; the four input CSV hashes are recorded in the report.\\n            base=[z[\'dev\'],z[\'test\']]\\n            baseline_source=\'user-authorized cached v3 predictions\'\\n        else:\\n            base=baseline_predict(train,frames)\\n            baseline_source=\'freshly fitted from train on hosted Colab\'\\n        alpha=float(selection[\'new_model_weight\'])\\n        status(\'new_model\',family=family,config=selection[\'config\'])\\n        if alpha==0:\\n            new=base\\n        elif family==\'linear\':\\n            new=linear_fit_predict(train,frames,selection[\'config\'])\\n        elif family==\'cat\':\\n            m=make_rich_cat();m.fit(rich_cat_features(train),train.spam_label.to_numpy(),verbose=300)\\n            new=[m.predict_proba(rich_cat_features(d),ntree_end=selection[\'config\'][\'trees\']) for d in frames]\\n        elif family==\'fasttext\':\\n            from fasttext_scratch import fit_predict\\n            new=fit_predict(train,frames,selection[\'config\'],out/\'fasttext_fit\')\\n        elif family==\'temperature\':\\n            cfg=selection[\'config\'];mix=cfg[\'cat_weight\'];temp=cfg[\'temperature\']\\n            new=[(1-mix)*softmax(temp*s,axis=1)+mix*p for s,p in zip(margins,cat_values)]\\n        elif family==\'stacking\':\\n            from stacking_v5 import fit_meta,features\\n            meta=fit_meta(features(meta_svm,meta_cat),train.spam_label.to_numpy(),selection[\'config\'])\\n            new=[meta.predict_proba(features(s,p)) for s,p in zip(margins,cat_values)]\\n        else:\\n            raise ValueError(\'Unknown model family\')\\n        probs=[(1-alpha)*a+alpha*b for a,b in zip(base,new)]\\n        w=np.asarray(selection[\'weights\'])\\n        dp,tp=[(p*w).argmax(1) for p in probs]\\n        sub=data[\'samplesubmission\'].copy()\\n        lookup=pd.Series(tp,index=test.id.astype(str))\\n        sub[\'spam_label\']=sub.ID.astype(str).map(lookup)\\n        if sub.spam_label.isna().any() or not sub.spam_label.isin([0,1,2,3]).all():\\n            raise ValueError(\'Invalid prediction IDs or labels\')\\n        sub.spam_label=sub.spam_label.astype(int)\\n        path=out/\'submission_v5.csv\';sub.to_csv(path,index=False)\\n        saved=pd.read_csv(path)\\n        pd.testing.assert_frame_equal(saved,sub,check_dtype=False)\\n        assert len(saved)==len(test) and saved.ID.equals(data[\'samplesubmission\'].ID)\\n        np.savez_compressed(out/\'predictions.npz\',dev=probs[0],test=probs[1],weights=w)\\n        report={\'selection\':selection,\'new_family\':family,\\n            \'dev\':metrics(dev.spam_label.to_numpy(),dp),\\n            \'v3_dev\':metrics(dev.spam_label.to_numpy(),base[0].argmax(1)),\\n            \'fit_data\':\'train only; no dev or test labels for selection/fitting\',\\n            \'pretrained\':False,\'execution\':\'hosted Google Colab CPU\',\\n            \'libraries\':{\'scikit-learn\':\'1.7.2 (BSD-3-Clause)\',\'CatBoost\':\'1.2.10 (Apache-2.0)\'},\\n            \'baseline_source\':baseline_source,\'rows_verified\':len(saved),\\n            \'prediction_counts\':{int(k):int(v) for k,v in saved.spam_label.value_counts().items()},\\n            \'data_sha256\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(data_dir).glob(\'ds102_*.csv\')},\\n            \'caveat\':\'OOF used repeatedly for model selection. Calibration cross-validation is on existing OOF, not a fully nested refit. Dev also reused in development. Kaggle score is unknown.\'}\\n        if family==\'fasttext\' and alpha>0:\\n            report[\'libraries\'][\'fastText\']=\'official v0.9.2 C++ CLI (MIT); random initialization\'\\n        (out/\'metrics_v5.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2))\\n        status(\'complete\',dev_macro_f1=report[\'dev\'][\'macro_f1\'],rows=len(saved))\\n        return report\\n    except BaseException as exc:\\n        import traceback\\n        status(\'error\',error=str(exc),traceback=traceback.format_exc())\\n        raise\\n"}')
for name,content in modules.items():
    (source/name).write_text(content)
sys.path.insert(0,str(source))


In [ ]:
from finalize_v5 import main
import json
chosen=json.loads('{"family": "temperature", "selection": {"raw_macro_f1": 0.7651232845255038, "cross_weight_macro_f1": 0.7681613908279921, "fold_deltas": [0.000904925851664351, 0.0022213587568191695, 0.0064716452158233295], "fold_weights": [[1, 1.2, 1.2, 1], [1, 1, 1.2, 0.9], [1, 1, 1.2, 0.9]], "use_weights": true, "weights": [1, 1, 1.2, 0.9], "selection_score": 0.7681613908279921, "config": {"temperature": 3, "cat_weight": 0.5}, "family": "temperature", "new_model_weight": 1, "model_index": 0, "raw_fold_f1": [0.7756573287215651, 0.761976967519318, 0.7570169883016922]}}')
report=main(chosen["selection"],chosen["family"],data_dir=str(data_dir),reference_path=None)
print("Dev F1-macro:",report["dev"]["macro_f1"])


In [ ]:
from google.colab import files
files.download('/content/ds102_v5/submission_v5.csv')
files.download('/content/ds102_v5/metrics_v5.json')
